# V11 THINK VERY HARD FINAL — Fixes Dust + Spread Loss + Cooldown

**V10 analysis (23 iters you ran):**
- [1] BUY $11.23 @ $78785
- [2] P/L 0.147% >= TP 0.140% → SELL 50% 0.000069669 → first quick TP
- [9]-[23] 15x dust sells 0.000100306 → 0.000000006, P/L 0.203%,0.205%,0.143%,0.210%,0.176%,0.205%,0.291%,0.353%,0.367%,0.328%,0.360%,0.304%,0.336%,0.369%,0.359% >= TP 0.14-0.25%
- Portfolio $93.53 → $93.33 -$0.20 despite 15 wins → spread > TP

**V11 fixes:**
1. Dust: qty<0.00008 sell 100% not 50%
2. Fee-aware TP min 0.18% = vol*1.8 + 0.08% spread
3. Cooldown 3 iters (15min) after sell
4. P/L>0.25% sell ALL to lock profit
5. Spacing 0.25% + falling knife filter slope<-0.15% RSI<25


In [ ]:
!pip -q install alpaca-py pandas matplotlib numpy
import time
from datetime import datetime, timedelta
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from getpass import getpass
print("V11 deps ready")


In [ ]:
ALPACA_API_KEY = getpass("Paste Alpaca Paper API KEY: ").strip()
ALPACA_SECRET = getpass("Paste FULL Secret: ").strip()
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce
from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.data.requests import CryptoBarsRequest, CryptoLatestQuoteRequest
from alpaca.data.timeframe import TimeFrame
trading_client=TradingClient(ALPACA_API_KEY, ALPACA_SECRET, paper=True)
data_client=CryptoHistoricalDataClient()
acct=trading_client.get_account()
print(f"✅ CONNECTED Portfolio ${float(acct.portfolio_value):.2f} BP ${float(acct.buying_power):.2f} Cash ${float(acct.cash):.2f}")


In [ ]:

def get_btc_price_5m():
    end=datetime.now()
    start=end-timedelta(days=2)
    req=CryptoBarsRequest(symbol_or_symbols="BTC/USD", timeframe=TimeFrame.Minute, start=start, end=end)
    bars=data_client.get_crypto_bars(req)
    df=bars.df
    if isinstance(df.index, pd.MultiIndex):
        try: df=df.xs("BTC/USD", level=0)
        except:
            try: df=df.xs("BTC/USD", level=1)
            except: df=df.droplevel(0)
    df=df.sort_index()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index=pd.to_datetime(df.index)
    df5=df["close"].resample("5min").ohlc()
    closes=df5["close"].dropna().tolist()
    if len(closes)<30:
        closes=df["close"].tolist()[-100:]
    return float(closes[-1]), closes

def compute_rsi(prices, period=14):
    if len(prices)<period+1: return 50.0
    delta=pd.Series(prices).diff()
    gain=delta.where(delta>0,0).rolling(period).mean()
    loss=-delta.where(delta<0,0).rolling(period).mean()
    rs=gain/loss
    rsi=100-(100/(1+rs))
    return float(rsi.iloc[-1]) if not pd.isna(rsi.iloc[-1]) else 50.0

def compute_sma(prices, period=20):
    if len(prices)<period: return float(prices[-1])
    return float(pd.Series(prices).rolling(period).mean().iloc[-1])

def compute_bollinger(prices, period=20, std=2):
    if len(prices)<period: return float(prices[-1]), float(prices[-1])*1.05, float(prices[-1])*0.95, 0.0, 0.0
    s=pd.Series(prices)
    sma=s.rolling(period).mean().iloc[-1]
    sd=s.rolling(period).std().iloc[-1]
    if pd.isna(sd): sd=s.std()
    upper=sma+std*sd
    lower=sma-std*sd
    if len(s)>=30:
        sma10=s.rolling(period).mean().iloc[-11:-1].mean()
        slope=(sma-sma10)/sma10*100 if sma10 else 0
    else:
        slope=0
    return float(sma), float(upper), float(lower), float(sd), float(slope)

def get_real_ofi():
    try:
        req=CryptoLatestQuoteRequest(symbol_or_symbols="BTC/USD")
        quotes=data_client.get_crypto_latest_quote(req)
        q=quotes.get("BTC/USD") if isinstance(quotes, dict) else quotes
        if isinstance(q, dict):
            bid_size=q.get("bid_size"); ask_size=q.get("ask_size")
        else:
            bid_size=getattr(q, "bid_size", None); ask_size=getattr(q, "ask_size", None)
        if bid_size and ask_size and (bid_size+ask_size)>0:
            ofi=(bid_size-ask_size)/(bid_size+ask_size)
            return float(ofi), True, float(bid_size), float(ask_size)
        return 0.0, False, 0.0, 0.0
    except:
        return 0.0, False, 0.0, 0.0

class MarketMemory:
    def __init__(self):
        self.pnl_history=[]
        self.vol_history=[]
    def add(self, pnl, vol):
        if pnl!=0: self.pnl_history.append(pnl)
        self.vol_history.append(vol)
        if len(self.pnl_history)>60: self.pnl_history=self.pnl_history[-60:]
        if len(self.vol_history)>60: self.vol_history=self.vol_history[-60:]
    def get_adaptive(self, base_vol, pnl, lots_len):
        recent_max = max(self.pnl_history[-20:]) if len(self.pnl_history)>=5 else 0.25
        avg_vol = float(np.mean(self.vol_history[-20:])) if self.vol_history else base_vol
        if avg_vol < 0.10:
            tp = max(0.18, min(0.40, recent_max*0.7 if recent_max>0.10 else avg_vol*2.0 + 0.08))
            sl = min(-0.22, max(-0.70, -avg_vol*2.5 - 0.05))
            regime=f"ULTRA_LOW vol {avg_vol:.3f}% TP_fee_aware {tp:.3f}%"
        elif avg_vol < 0.20:
            tp = max(0.20, min(0.60, avg_vol*1.8 + 0.08))
            sl = min(-0.28, max(-1.0, -avg_vol*2.3 - 0.05))
            regime=f"LOW vol {avg_vol:.3f}%"
        else:
            tp = max(0.30, min(1.0, avg_vol*1.6 + 0.08))
            sl = min(-0.40, max(-1.5, -avg_vol*2.0 - 0.05))
            regime=f"NORMAL vol {avg_vol:.3f}%"
        if lots_len >= 5:
            tp = tp*0.75
            regime+="+CAP-25%"
        return tp, sl, regime

market_mem = MarketMemory()
for pnl in [0.147,0.078,0.203,0.205,0.143,0.210,0.176,0.205,0.291,0.353,0.367,0.328,0.360,0.304,0.336,0.369,0.359]:
    market_mem.add(pnl, 0.15)

class Trader:
    def __init__(self, client):
        self.client=client
        self.lots=[]
        self.total=0.0
        self.max_buys=5
        self.max_exposure_usd=60.0
        self.last_sell_iter=-10
        self.refresh()
    def refresh(self):
        try:
            for p in self.client.get_all_positions():
                if "BTC" in p.symbol:
                    self.total=float(p.qty)
                    if not self.lots and self.total>0:
                        price,_=get_btc_price_5m()
                        self.lots=[(price, self.total)]
        except: pass
    def get_avg_entry(self):
        if not self.lots: return 0.0
        tq=sum(q for _,q in self.lots)
        return sum(p*q for p,q in self.lots)/tq if tq>0 else 0.0
    def get_current_exposure(self, price):
        return self.total*price
    def buy(self, bp, iteration, reason):
        if iteration - self.last_sell_iter < 3:
            print(f"  ⏸️ BUY blocked cooldown {iteration - self.last_sell_iter}/3 after sell")
            return None
        if bp<10.10: return None
        try:
            cp,_=get_btc_price_5m()
            exp=self.get_current_exposure(cp)
            if len(self.lots)>=self.max_buys: return None
            if exp>=self.max_exposure_usd: return None
            dollars=round(max(10.10, min(bp*0.12, 12.0)),2)
            rem=self.max_exposure_usd-exp
            if rem<10.10: return None
            dollars=min(dollars, round(rem,2))
            from alpaca.trading.requests import MarketOrderRequest
            from alpaca.trading.enums import OrderSide, TimeInForce
            order=MarketOrderRequest(symbol="BTC/USD", notional=dollars, side=OrderSide.BUY, time_in_force=TimeInForce.GTC)
            res=self.client.submit_order(order)
            price,_=get_btc_price_5m()
            qty=dollars/price
            self.lots.append((price, qty))
            print(f"  ✅ BUY ${dollars:.2f} @ ${price:.2f} qty {qty:.9f} order {res.id} avg ${self.get_avg_entry():.2f} {reason}")
            time.sleep(5); self.refresh()
            return res
        except Exception as e:
            print(f"  ❌ BUY FAILED {e}")
            return None
    def sell(self, reason, iteration, force_all=False):
        try:
            avail=0
            for p in self.client.get_all_positions():
                if "BTC" in p.symbol:
                    avail=float(p.qty_available)
            if avail<=0: return None
            if avail < 0.00008 or force_all:
                qty=float(f"{avail:.9f}")
                all_flag=True
            else:
                qty=float(f"{(avail*0.5):.9f}")
                all_flag=False
            from alpaca.trading.requests import MarketOrderRequest
            from alpaca.trading.enums import OrderSide, TimeInForce
            order=MarketOrderRequest(symbol="BTC/USD", qty=qty, side=OrderSide.SELL, time_in_force=TimeInForce.GTC)
            res=self.client.submit_order(order)
            price,_=get_btc_price_5m()
            print(f"  ✅ SELL {reason} qty {qty:.9f} @ ${price:.2f} order {res.id} {'ALL' if all_flag else '50%'}")
            rem=qty
            while rem>1e-12 and self.lots:
                lp,lq=self.lots[0]
                if lq<=rem+1e-12:
                    rem-=lq; self.lots.pop(0)
                else:
                    self.lots[0]=(lp,lq-rem); rem=0
            self.last_sell_iter=iteration
            time.sleep(5); self.refresh()
            return res
        except Exception as e:
            print(f"  ❌ SELL FAILED {e}")
            return None

trader=Trader(trading_client)
price, closes = get_btc_price_5m()
rsi=compute_rsi(closes)
sma,_,_,sd,slope=compute_bollinger(closes)
ofi, is_real, bs, ask_s = get_real_ofi()
print(f"✅ READY price ${price:.2f} RSI {rsi:.1f} SMA ${sma:.2f} slope {slope:.4f}% OFI {ofi:.4f} BTC {trader.total:.9f} avg ${trader.get_avg_entry():.2f}")


In [ ]:

DURATION_HOURS=6
INTERVAL=300
start=time.time()
end=start+DURATION_HOURS*3600
iteration=0
balances=[]

print(f"=== V11 FINAL START {datetime.now()} ===")
print(f"Dust fix <0.00008 sell ALL, fee-aware TP min 0.18%, cooldown 15min, 0.25%+ sell ALL")

try:
    while time.time() < end:
        iteration+=1
        price, closes = get_btc_price_5m()
        rsi=compute_rsi(closes)
        sma20,upper,lower,sd,slope=compute_bollinger(closes)
        ofi_real, is_real, bs, ask_s = get_real_ofi()
        base_vol = (sd/sma20*100) if sma20 else 0.15
        avg=trader.get_avg_entry() if trader.lots else price
        pnl=(price-avg)/avg*100 if avg else 0

        market_mem.add(pnl, base_vol)
        tp_pct, sl_pct, regime = market_mem.get_adaptive(base_vol, pnl, len(trader.lots))

        if slope < -0.15 and rsi < 25 and price < lower:
            blocked=f"FALLING_KNIFE slope {slope:.3f}% RSI {rsi:.1f} price<lower"
            score=0
        else:
            cond_rsi = rsi < 45
            cond_sma = price < sma20
            cond_lower = price <= lower*1.005
            cond_ofi = abs(ofi_real) > 0.003
            score = int(cond_rsi) + int(cond_sma) + int(cond_lower) + int(cond_ofi)
            blocked=""

        buy_thresh = 2 if base_vol < 0.20 else 3
        if rsi < 20:
            buy_thresh=4

        spacing_ok=True
        if avg and abs(price-avg)/avg*100 < 0.25 and len(trader.lots)>=2:
            spacing_ok=False

        acct=trading_client.get_account()
        bp=float(acct.buying_power)
        pv=float(acct.portfolio_value)

        print(f"[{iteration}] {regime} slope {slope:.3f}% RSI {rsi:.1f} price {price:.2f} score {score}/4 thresh {buy_thresh} vol {base_vol:.3f}% TP {tp_pct:.3f}% SL {sl_pct:.3f}% P/L {pnl:.3f}% {blocked}")

        if pnl>=tp_pct and trader.total>0:
            force_all = pnl > 0.25
            print(f"  -> TP {pnl:.3f}% >= {tp_pct:.3f}% {'FORCE_ALL>0.25%' if force_all else ''}")
            trader.sell(f"TP {pnl:.3f}% >= {tp_pct:.3f}% {regime}", iteration, force_all=force_all)
        elif pnl<=sl_pct and trader.total>0:
            print(f"  -> SL {pnl:.3f}% <= {sl_pct:.3f}%")
            trader.sell(f"SL {pnl:.3f}% <= {sl_pct:.3f}% {regime}", iteration, force_all=True)
        elif score>=buy_thresh and blocked=="" and spacing_ok:
            if len(trader.lots)>=trader.max_buys:
                print(f"  -> BUY signal but SKIP cap {trader.max_buys}")
            elif iteration - trader.last_sell_iter < 3:
                print(f"  -> BUY signal but cooldown {iteration - trader.last_sell_iter}/3")
            else:
                print(f"  -> BUY signal")
                trader.buy(bp, iteration, f"score {score}/4 {regime}")
        else:
            print(f"  -> HOLD {blocked if blocked else ''} {'spacing<0.25%' if not spacing_ok else ''}")

        acct=trading_client.get_account()
        pv=float(acct.portfolio_value)
        bp=float(acct.buying_power)
        balances.append(pv)
        print(f"  Portfolio ${pv:.2f} BP ${bp:.2f} BTC {trader.total:.9f} avg ${trader.get_avg_entry():.2f} rem {(end-time.time())/60:.1f}m\n")
        time.sleep(min(INTERVAL, end-time.time()))
except KeyboardInterrupt:
    print("Interrupted")
print("FINISHED")
acct=trading_client.get_account()
positions=trading_client.get_all_positions()
print(f"Final Portfolio ${float(acct.portfolio_value):.2f}")
for p in positions:
    print(f"  {p.symbol} qty {p.qty} market ${float(p.market_value):.2f} P/L ${float(p.unrealized_pl):.4f}")
plt.figure(figsize=(12,5))
plt.plot(balances, marker="o")
plt.title(f"V11 Think Hard Final - {iteration} iters")
plt.ylabel("Portfolio $")
plt.grid(True)
plt.savefig("v11_final.png")
plt.show()
